# Notebook 01 — Reproducción del ejemplo de Sigrist (2021)

Replicamos el ejemplo simulado del artículo *Tree-Boosting for Spatial Data* (TDS Archive, 2021): $n=200$ observaciones generadas por $y = F(x_1) + b(s) + e$ con covarianza exponencial ($\sigma^2_{GP}=0{,}35$, $\rho=0{,}1$) y nugget $\sigma^2=0{,}1$. Entrenamos GPBoost, predecimos sobre rejilla 50×50, generamos la figura 1 del artículo y comparamos contra Linear GP (universal kriging), L2Boost y Random Forest.

## Compatibilidad con Google Colab

La siguiente celda detecta si el notebook se ejecuta en Colab y, en ese caso, instala `gpboost` y `shap` y clona el repositorio para tener acceso a los datos derivados (`data/`) y a los modelos entrenados (`modelos/`). En Jupyter local no hace nada.

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Entorno Colab — instalando dependencias...')
    !pip -q install gpboost shap
    if not os.path.isdir('/content/repo'):
        !git clone -q https://github.com/linaq11/examen-2-gpboost.git /content/repo
    os.chdir('/content/repo')
    print('CWD:', os.getcwd())
else:
    # En local: asume que el notebook está en repo/notebooks/
    here = os.path.dirname(os.path.abspath('__file__')) if '__file__' in globals() else os.getcwd()
    if os.path.basename(here) == 'notebooks':
        os.chdir(os.path.dirname(here))
    print('CWD:', os.getcwd())

## 1. Setup

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gpboost as gpb

np.random.seed(1)
DATA = 'data'
MOD  = 'modelos'
FIG  = 'figuras'
os.makedirs(FIG, exist_ok=True)
from sklearn.ensemble import RandomForestRegressor

## 2. Carga de los datos simulados (semilla=1, reproducibles)

Los datos se generaron una sola vez con la semilla `np.random.seed(1)` siguiendo el apéndice del artículo y se persistieron en `data/simulado_train_seed1.csv`.

In [ ]:
df_train = pd.read_csv(os.path.join(DATA, 'simulado_train_seed1.csv'))
X      = df_train[['X1','X2']].values
y      = df_train['y'].values
coords = df_train[['s_1','s_2']].values
n      = len(y)
print(f'n={n}, X.shape={X.shape}, coords.shape={coords.shape}')
df_train.head()

## 3. Visualización de los datos simulados

Tres paneles: la función verdadera $F(x_1)$, el campo $b(s)$ y las observaciones $y$ proyectadas sobre el plano espacial.

In [ ]:
def f1d(x):
    x = np.asarray(x, dtype=float).copy()
    x[np.abs(x - 0.5) < 1e-12] = 0.5 + 1e-10
    return np.sin(3*np.pi*x) + (1 + 3*np.maximum(np.zeros_like(x), x-0.5)/(x-0.5)) - 3

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
xx = np.linspace(0, 1, 200)
axes[0].plot(xx, f1d(xx), 'k-', lw=2)
axes[0].set_xlabel('x_1'); axes[0].set_ylabel('F(x_1)')
axes[0].set_title('a) Función no-lineal F(x_1)'); axes[0].grid(alpha=0.3)

sc = axes[1].scatter(coords[:,0], coords[:,1], c=df_train['b'].values,
                     cmap='RdBu_r', s=30); plt.colorbar(sc, ax=axes[1])
axes[1].set_xlabel('s_1'); axes[1].set_ylabel('s_2')
axes[1].set_title('b) Campo b(s) verdadero')

sc = axes[2].scatter(coords[:,0], coords[:,1], c=y, cmap='viridis', s=30)
plt.colorbar(sc, ax=axes[2])
axes[2].set_xlabel('s_1'); axes[2].set_ylabel('s_2')
axes[2].set_title('c) Observaciones y')

plt.tight_layout()
plt.savefig(os.path.join(FIG, 'fig1_datos_simulados.png'), dpi=120, bbox_inches='tight')
plt.show()

## 4. Entrenamiento GPBoost

Hiperparámetros del artículo: `learning_rate=0.01`, `max_depth=3`, `min_data_in_leaf=10`, `num_leaves=1024`, $M=247$ iteraciones, covarianza exponencial.

In [ ]:
# Reentrenamos el modelo (es rápido, ~3 s) para que el notebook sea autocontenido.
gp_model = gpb.GPModel(gp_coords=coords, cov_function='exponential')
params = {'objective':'regression_l2','learning_rate':0.01,'max_depth':3,
          'min_data_in_leaf':10,'num_leaves':1024,'verbose':-1}
ds = gpb.Dataset(X, y)
bst = gpb.train(params=params, train_set=ds, gp_model=gp_model,
                num_boost_round=247)
bst.save_model(os.path.join(MOD, 'gpboost_basico.json'))
print('Modelo guardado en', os.path.join(MOD, 'gpboost_basico.json'))

## 5. Predicción sobre rejilla 50×50 y figura 1 del artículo

In [ ]:
nx = 50
s1x = np.linspace(0,1,nx); s2x = np.linspace(0,1,nx)
S1, S2 = np.meshgrid(s1x, s2x)
coords_test = np.column_stack([S1.flatten(), S2.flatten()])
X_test = np.column_stack([np.linspace(0,1,nx*nx), np.linspace(0,1,nx*nx)])

pred = bst.predict(data=X_test, gp_coords_pred=coords_test,
                   pred_latent=True, predict_var=True)
F_pred = np.array(pred['fixed_effect']).reshape(nx, nx)
b_pred = np.array(pred['random_effect_mean']).reshape(nx, nx)
b_var  = np.array(pred['random_effect_cov']).reshape(nx, nx)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
ax = axes[0,0]; im = ax.imshow(b_pred, origin='lower', extent=[0,1,0,1], cmap='RdBu_r')
plt.colorbar(im, ax=ax); ax.set_title('GP predicho b̂(s)')
ax = axes[0,1]; im = ax.imshow(np.sqrt(b_var), origin='lower', extent=[0,1,0,1], cmap='magma')
plt.colorbar(im, ax=ax); ax.set_title('SD posterior del GP')
ax = axes[1,0]
xx = np.linspace(0,1,200)
X_curve = np.column_stack([xx, np.full_like(xx, 0.5)])
F_curve = np.array(bst.predict(data=X_curve, pred_latent=True, ignore_gp_model=True)).flatten()
ax.plot(xx, f1d(xx), 'k-', lw=2, label='F verdadero')
ax.plot(xx, F_curve, 'r--', lw=2, label='F̂ GPBoost')
ax.legend(); ax.set_xlabel('x_1'); ax.set_title('F̂(x_1) vs verdadero')
ax = axes[1,1]
sc = ax.scatter(coords[:,0], coords[:,1], c=df_train['b'].values, cmap='RdBu_r', s=30)
plt.colorbar(sc, ax=ax); ax.set_title('b(s) verdadero (referencia)')
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'fig2_predicciones_gpboost.png'), dpi=120, bbox_inches='tight')
plt.show()

## 6. Comparativa con métodos alternativos

Sobre los mismos 200 puntos de entrenamiento y un test grid 50×50 reproducible comparamos: Linear GP (universal kriging con $F(x)$ lineal), L2Boost (tree-boosting con coordenadas como features adicionales) y Random Forest con coordenadas.

In [ ]:
# Para evaluar RMSE necesitamos un test set con los mismos parámetros pero independiente.
# Usamos el conjunto comparativo ya derivado:
tabla = pd.read_csv(os.path.join(DATA, 'tabla_comparativa_metodos.csv'))
tabla

### Lectura

GPBoost obtiene el menor RMSE de los cuatro, ordenado igual que en el artículo original. El Linear GP queda muy por detrás porque su componente fijo lineal no puede capturar la discontinuidad de $F(x_1)$. L2Boost y Random Forest se acercan pero no superan a GPBoost: introducir las coordenadas como features no es equivalente a modelar explícitamente el proceso espacial.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(tabla['Método'], tabla['RMSE'], color=['#2E86AB','#A23B72','#F18F01','#C73E1D'])
for i, v in enumerate(tabla['RMSE']):
    ax.text(v+0.01, i, f'{v:.4f}', va='center')
ax.set_xlabel('RMSE sobre test grid 50×50')
ax.set_title('Comparativa de métodos sobre la simulación de Sigrist 2021')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'fig3_comparativa_metodos.png'), dpi=120, bbox_inches='tight')
plt.show()

## 7. Interpretación con SHAP

Valor SHAP del booster (componente fijo). Solo $X_1$ tiene señal por construcción; $X_2$ debería tener importancia cercana a cero.

In [ ]:
import shap
explainer = shap.TreeExplainer(bst)
shap_values = explainer.shap_values(X)
shap.summary_plot(shap_values, X, feature_names=['X1','X2'], show=False)
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'fig4_shap_values.png'), dpi=120, bbox_inches='tight')
plt.show()

---

**Conclusión Fase 1.** El ejemplo de Sigrist 2021 se reproduce de forma satisfactoria: GPBoost recupera la forma de $F(x_1)$ y el campo $b(s)$, SHAP atribuye casi toda la importancia a $X_1$, y la tabla comparativa preserva el orden cualitativo del artículo. Los notebooks 02–05 profundizan en la dependencia espacial.